# Monitor the training process -- Weather Prediction (PyTorch)

We work on a regression task (weather prediction):
- how to create a neural network
- how to optimize the model
- how to monitor the training process
- how to detect (or avoid) overfitting
- common options to improve the model performance

## 0. Import packages and modules

Enable Array-API support globally to reduce need for data movement between devices and for scikit-learn to better support PyTorch tensors.

In [ ]:
import os

#os.environ["SCIPY_ARRAY_API"] = "1"

import sklearn
#sklearn.set_config(array_api_dispatch=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn

import torch
import torch.nn as nn

print(f"PyTorch version: {torch.__version__}")
print(f"Sklearn version: {sklearn.__version__}")

# Check if GPU is available
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

## 1.Formulate/Outline the problem: Weather Prediction

<img src="https://learn.mimer-ai.eu/deep-learning-intro/_images/03_weather_prediction_dataset_map.png" alt="drawing" width="720"/>

## 2. Identify inputs and outputs

In [ ]:
data = pd.read_csv("https://zenodo.org/record/5071376/files/weather_prediction_dataset_light.csv?download=1")
data

In [ ]:
data.describe()

In [ ]:
data.columns

In [ ]:
data.shape

## 3. Prepare data

### 3.1 Select a subset and split into data and y_batch

In [ ]:
nr_rows = 365*3 # 3 years

# data
X_data = data.loc[:nr_rows] # Select first 3 years
X_data = X_data.drop(columns=['DATE', 'MONTH']) # Drop date and month column

# y_batch (sunshine hours the next day)
y_data = data.loc[1:(nr_rows + 1)]["BASEL_sunshine"]

### 3.2 Split data and y_batch into training, validation, and test set

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_holdout, y_train, y_holdout = train_test_split(X_data, y_data, test_size=0.3, random_state=0)
X_val, X_test, y_val, y_test = train_test_split(X_holdout, y_holdout, test_size=0.5, random_state=0)

### 3.3 Prepare datasets and dataloaders (PyTorch only)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader


train_dataset = TensorDataset(
    torch.tensor(X_train.values, dtype=torch.float),
    torch.tensor(y_train.values, dtype=torch.float)
)
# NOTE: we will use the validation dataset later
test_dataset = TensorDataset(
    torch.tensor(X_test.values, dtype=torch.float),
    torch.tensor(y_test.values, dtype=torch.float)
)


train_dl = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dl = DataLoader(test_dataset, batch_size=32, shuffle=False)

## 4. Building architecture from scratch

### 4.1 Build the neuron network

In [ ]:
import torch.nn.functional as F


class WeatherPredictionModel(nn.Module):
    def __init__(self, input_shape, hidden1=100, hidden2=50):
        super().__init__()

        self.hidden_layer1 = nn.Linear(input_shape, hidden1)
        self.hidden_layer2 = nn.Linear(hidden1, hidden2)
        self.output_layer = nn.Linear(hidden2, 1)
    
    def forward(self, x):
        x = self.hidden_layer1(x)
        x = F.relu(x)
        x = self.hidden_layer2(x)
        x = F.relu(x)
        x = self.output_layer(x)
        return x

model = WeatherPredictionModel(X_train.shape[1])

In [ ]:
# summary of the model
print(model)

In [ ]:
from torchinfo import summary

example_batch_size = 32
summary(model, input_size=(example_batch_size, X_train.shape[1]))

### 4.2 Gradient descent

<img src="https://editor.analyticsvidhya.com/uploads/93873gd1.png" alt="drawing" width="400"/> <img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*HJifIvvsZpQdGEenC8w-Qg.jpeg" alt="drawing" width="350"/>

**Gradient Descent** (GD) is an optimization algorithm used to find the minimum of a function
- `gradient` means an increase and decrease in a property (LOSS function)
- `descent` means the action of moving downwards
- GD means to update the parameters (weights and biases) against the direction of the gradient, so as to find the minimum of the dependent function (the loss function)

**GD** has lots of variations
- **Stochastic Gradient Descent (SGD)**
    - SGD only uses a one sample of the training dataset to update the model parameters
- **Batch Gradient Descent**
    - it computes the gradient of loss function using entire training dataset in one epoch
- **Mini-Batch Gradient Descent**, a variant of SGD, is commonly used
    - gradients are computed over small subsets of data to combine the benefits of both methods—stability from batch processing and efficiency from stochastic updates
    - this subset is called `batch`
    - the number of samples in one batch is called `batch size` (*e.g.*, 32 or 64 samples)

<img src="https://enccs.github.io/deep-learning-intro/_images/03_gradient_descent.png" alt="drawing" width="600"/>

**learning rate** is a parameter in GD algorithm
- it controls how big a step we should take to decrease a property (LOSS function)
- if it is too small, training is very slow and might get stuck in local minima
- it it is too large, training might overshoot the minimum or even diverge
- a better way is to adjust learning rate over time
- the **Adam** optimizer (Adaptive Moment Estimation)
    - it combines momentum + adaptive learning rate
    - momentum means to add a velocity term to smooth updates and escape local minima

## 5. Choose a loss function and optimizer

In [ ]:
# loss function, optimizer
import torch.optim as optim

optimizer = optim.Adam(model.parameters())
loss_fn = nn.MSELoss()

## 6. Train the model

In [ ]:
from tqdm import tqdm
from sklearn import metrics


def train_epoch(model, data_loader, loss_fn, optimizer, progress_desc):
    """Training the model for one epoch."""
    model.train()
    running_loss = 0.0
    running_rmse = 0.0

    if progress_desc:
        pbar = tqdm(data_loader, desc=progress_desc)
    else:
        pbar = data_loader

    for i, (X_batch, y_batch) in enumerate(pbar):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Zero the gradients
        optimizer.zero_grad()
  
        # Forward pass and make predictions
        y_pred = model(X_batch)

        # Compute loss
        loss = loss_fn(y_pred.squeeze(), y_batch)
        
        # Backward pass and update weights using the optimizer
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_rmse += metrics.root_mean_squared_error(y_batch.detach().to('cpu'), y_pred.detach().to('cpu'))
        if progress_desc:
            pbar.set_postfix({"loss": running_loss / (i + 1), "rmse": running_rmse / (i + 1)})
    
    train_loss = running_loss / len(data_loader)
    train_rmse = running_rmse / len(data_loader)
    # print(f"{train_loss = }, {train_rmse = }")
    return train_loss, train_rmse

def eval_epoch(model, data_loader, loss_fn, accumulate=False):
    """Evaluate the model for one epoch for testing / validation data. 
    No gradients are computed, no backpropagation.
    
    """
    model.eval()
    running_loss = 0.0
    running_rmse = 0.0
    y_true_tensor = torch.tensor([]).to(device)
    y_pred_tensor = torch.tensor([]).to(device)

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred = model(X_batch)
            loss = loss_fn(y_pred.squeeze(), y_batch)

            running_loss += loss.item()
            running_rmse += metrics.root_mean_squared_error(y_batch.detach().to('cpu'), y_pred.detach().to('cpu'))

            if accumulate:
                y_true_tensor = torch.cat((y_true_tensor, y_batch))
                y_pred_tensor = torch.cat((y_pred_tensor, y_pred))
    
    eval_loss = running_loss / len(data_loader)
    eval_rmse = running_rmse / len(data_loader)
    return eval_loss, eval_rmse, y_true_tensor.cpu(), y_pred_tensor.cpu()

In [ ]:
model = model.to(device)

history = {'loss': [], 'root_mean_squared_error': []}
epochs = 200

for epoch in range(epochs):
    loss, rmse = train_epoch(model, train_dl, loss_fn, optimizer, f"Epoch {epoch+1}/{epochs}")
    history['loss'].append(loss)
    history['root_mean_squared_error'].append(rmse)


In [ ]:
def plot_history(history, metrics):
    """
    Plot the training history

    Args:
        history (dict): Dictionary containing training history
        metrics (str, list): Metric or a list of metrics to plot
    """
    history_df = pd.DataFrame(history)
    sns.lineplot(data=history_df[metrics])
    plt.xlabel("epochs")
    plt.ylabel("metric")

plot_history(history, 'root_mean_squared_error')

## 7. Perform a prediction/classification

In [ ]:
_, train_rmse, y_train_true, y_train_predicted = eval_epoch(model, train_dl, loss_fn, accumulate=True)
_, test_rmse, y_test_true, y_test_predicted = eval_epoch(model, test_dl, loss_fn, accumulate=True)

## 8. Measure performance

### 8.1 Measure performance of the trained model

In [ ]:
def plot_predictions(y_pred, y_true, title):
    plt.style.use('ggplot')  # optional, that's only to define a visual style
    plt.scatter(y_pred, y_true, s=10, alpha=0.5)
    plt.axline((0,0),slope = 1, color = "black") # plot diagonal reference line
    plt.xlabel("predicted sunshine hours")
    plt.ylabel("true sunshine hours")
    plt.title(title)

plot_predictions(y_train_predicted, y_train_true, title='Predictions on the training set')

In [ ]:
plot_predictions(y_test_predicted, y_test_true, title='Predictions on the test set')

In [ ]:
print(f'Train RMSE: {train_rmse:.2f}, Test RMSE: {test_rmse:.2f}')

### 8.2 The baseline model

In [ ]:
# Baseline prediction using previous day's sunshine hours
y_baseline_prediction = X_test['BASEL_sunshine'].values
plot_predictions(y_baseline_prediction, y_test, title='Baseline predictions on the test set')

In [ ]:
from sklearn.metrics import root_mean_squared_error

rmse_baseline = root_mean_squared_error(y_test, y_baseline_prediction)
print('Baseline:', rmse_baseline)
print('Neural network: ', test_rmse)

## 9. Refine the model

### 9.1 The validation dataset

In [ ]:
val_dataset = TensorDataset(
    torch.tensor(X_val.values, dtype=torch.float),
    torch.tensor(y_val.values, dtype=torch.float)
)
val_dl = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
model = WeatherPredictionModel(input_shape=X_data.shape[1])
optimizer = optim.Adam(model.parameters())
loss_fn = nn.MSELoss()

In [ ]:
model = model.to(device)

history = {
    'loss': [],
    'root_mean_squared_error': [],
    'val_loss': [],
    'val_root_mean_squared_error': []
}
epochs = 200

for epoch in range(epochs):
    loss, rmse = train_epoch(model, train_dl, loss_fn, optimizer, False)
    history['loss'].append(loss)
    history['root_mean_squared_error'].append(rmse)

    val_loss, val_rmse, *_ = eval_epoch(model, val_dl, loss_fn)
    history['val_loss'].append(val_loss)
    history['val_root_mean_squared_error'].append(val_rmse)

In [ ]:
plot_history(history, ['root_mean_squared_error', 'val_root_mean_squared_error'])

### 9.2 Counteract model overfitting

#### 9.2.1 Reduce number of parameters

In [ ]:
# Create smaller model
model = WeatherPredictionModel(input_shape=X_data.shape[1], hidden1=10, hidden2=5)
summary(model, input_size=(32, X_data.shape[1]))

In [ ]:
optimizer = optim.Adam(model.parameters())
loss_fn = nn.MSELoss()

In [ ]:
model = model.to(device)

history = {
    'loss': [],
    'root_mean_squared_error': [],
    'val_loss': [],
    'val_root_mean_squared_error': []
}
epochs = 200

for epoch in range(epochs):
    loss, rmse = train_epoch(model, train_dl, loss_fn, optimizer, False)
    history['loss'].append(loss)
    history['root_mean_squared_error'].append(rmse)

    val_loss, val_rmse, *_ = eval_epoch(model, val_dl, loss_fn)
    history['val_loss'].append(val_loss)
    history['val_root_mean_squared_error'].append(val_rmse)

plot_history(history, ['root_mean_squared_error', 'val_root_mean_squared_error'])

#### 9.2.2 Early stopping: stop when things are looking best

In [ ]:
model = WeatherPredictionModel(input_shape=X_data.shape[1])
optimizer = optim.Adam(model.parameters())
loss_fn = nn.MSELoss()

In [ ]:
def fit(model, train_dl, loss_fn, optimizer, val_dl):
    model = model.to(device)

    history = {
        'loss': [],
        'root_mean_squared_error': [],
        'val_loss': [],
        'val_root_mean_squared_error': []
    }
    epochs = 200
    early_stopping_patience = 10

    best_val_loss = float('inf')
    patience_counter = 0
        
    for epoch in range(epochs):
        loss, rmse = train_epoch(
            model,
            train_dl,
            loss_fn,
            optimizer,
            f"Epoch {epoch+1}/{epochs} {best_val_loss=:.2f}"
        )
        history['loss'].append(loss)
        history['root_mean_squared_error'].append(rmse)

        val_loss, val_rmse, *_ = eval_epoch(model, val_dl, loss_fn)
        history['val_loss'].append(val_loss)
        history['val_root_mean_squared_error'].append(val_rmse)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # Save best model
            best_model_state = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f'Early stopping at epoch {epoch+1}')
                # Restore best model
                model.load_state_dict(best_model_state)
                break
        
    return model, history

model, history = fit(model, train_dl, loss_fn, optimizer, val_dl)

In [ ]:
plot_history(history, ['root_mean_squared_error', 'val_root_mean_squared_error'])

#### 9.2.3 BatchNorm: the `standard scaler` for deep learning

In [ ]:
class WeatherPredictionModelBatchNorm(nn.Module):
    def __init__(self, input_shape, hidden1=100, hidden2=50):
        super().__init__()
        self.batch_norm = nn.BatchNorm1d(input_shape)
        self.hidden_layer1 = nn.Linear(input_shape, hidden1)
        self.hidden_layer2 = nn.Linear(hidden1, hidden2)
        self.output_layer = nn.Linear(hidden2, 1)
    
    def forward(self, x):
        x = self.batch_norm(x)
        x = self.hidden_layer1(x)
        x = F.relu(x)
        x = self.hidden_layer2(x)
        x = F.relu(x)
        x = self.output_layer(x)
        return x

model = WeatherPredictionModel(X_train.shape[1])
summary(model, input_size=(32, X_train.shape[1]))

In [ ]:
optimizer = optim.Adam(model.parameters())
loss_fn = nn.MSELoss()

In [ ]:
model, history = fit(model, train_dl, loss_fn, optimizer, val_dl)

In [ ]:
plot_history(history, ['root_mean_squared_error', 'val_root_mean_squared_error'])

In [ ]:
_, test_rmse, y_test_true, y_test_predicted = eval_epoch(model, test_dl, loss_fn, accumulate=True)

In [ ]:
plot_predictions(y_test_predicted, y_test_true, title='Predictions on the test set')

In [ ]:
print('Baseline:', rmse_baseline)
print('Test RMSE:', test_rmse)

### 9.3 Simplify the model and add data

In [ ]:
cols = [c for c in X_data.columns if c[:5] == 'BASEL']
X_data = X_data[cols]

In [ ]:
# use 9 years out of the dataset

nr_rows = 365*9
# data
X_data = data.loc[:nr_rows].drop(columns=['DATE', 'MONTH'])

# y_batch (sunshine hours the next day)
y_data = data.loc[1:(nr_rows + 1)]["BASEL_sunshine"]

In [ ]:
# only use columns with 'BASEL'

cols = [c for c in X_data.columns if c[:5] == 'BASEL']
X_data = X_data[cols]

In [ ]:
# Rerun the model and evaluate it

X_train, X_holdout, y_train, y_holdout = train_test_split(X_data, y_data, test_size=0.3, random_state=0)
X_val, X_test, y_val, y_test = train_test_split(X_holdout, y_holdout, test_size=0.5, random_state=0)

In [ ]:
X_train.shape

In [ ]:
train_dataset = TensorDataset(
    torch.tensor(X_train.values, dtype=torch.float),
    torch.tensor(y_train.values, dtype=torch.float)
)
test_dataset = TensorDataset(
    torch.tensor(X_test.values, dtype=torch.float),
    torch.tensor(y_test.values, dtype=torch.float)
)
val_dataset = TensorDataset(
    torch.tensor(X_val.values, dtype=torch.float),
    torch.tensor(y_val.values, dtype=torch.float)
)

train_dl = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dl = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_dl = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
model = WeatherPredictionModelBatchNorm(input_shape=X_data.shape[1])
summary(model, (32, X_train.shape[1]))

In [ ]:
optimizer = optim.Adam(model.parameters())
loss_fn = nn.MSELoss()

In [ ]:
model, history = fit(model, train_dl, loss_fn, optimizer, val_dl)

In [ ]:
plot_history(history, ['root_mean_squared_error', 'val_root_mean_squared_error'])

In [ ]:
_, test_rmse, y_test_true, y_test_predicted = eval_epoch(model, test_dl, loss_fn, accumulate=True)

In [ ]:
plot_predictions(y_test_predicted, y_test_true, title='Predictions on the test set')

In [ ]:
print('Baseline:', rmse_baseline)
print('Test RMSE:', test_rmse)

### Using Tensorboard for experiment tracking

In [ ]:
# TensorBoard logging
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter


log_dir = "logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
writer = SummaryWriter(log_dir=log_dir)

def fit_with_tensorboard(model, train_loader, val_loader, loss_fn, optimizer):
    model = model.to(device)
    val_loss = float("inf")
    epochs = 50

    for epoch in range(epochs):
        # Training phase
        loss, rmse = train_epoch(
            model,
            train_dl,
            loss_fn,
            optimizer,
            f"Epoch {epoch+1}/{epochs} {val_loss=:.2f}"
        )
        writer.add_scalar('training loss', loss, epoch)

        # Validation phase
        val_loss, val_rmse, *_ = eval_epoch(model, val_loader, loss_fn)

        # Log to TensorBoard
        writer.add_scalar('validation loss', val_loss, epoch)
            
    writer.close()
    return model

In [ ]:
model = WeatherPredictionModelBatchNorm(input_shape=X_data.shape[1])
summary(model, (32, X_train.shape[1]))

In [ ]:
optimizer = optim.Adam(model.parameters())
loss_fn = nn.MSELoss()

In [ ]:
model = fit_with_tensorboard(model, train_dl, val_dl, loss_fn, optimizer)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

## 10. Save the model

In [ ]:
trained_model_path = "weather_prediction_model_pytorch.pt"
torch.save(model_final.state_dict(), trained_model_path)
print(f"Model saved to {trained_model_path}")